# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process a dataset described by a [Croissant schema](https://mlcommons.org/croissant/) using the `mlcroissant` library. All data entities are referenced using their `@id` fields as required by the schema.

### Dataset Source

The dataset Croissant schema is available at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

Let's review available record sets and fields defined in the dataset schema. All references are by `@id`.

_Note: `mlcroissant` exposes record set and field metadata using the `dataset.metadata.record_sets` property._

In [ ]:
print("Available record sets in this dataset (referenced by @id):\n")
for recset in metadata.record_sets:
    print(f"  * Record set @id: {recset.id}   (Name: {recset.name})")
    if hasattr(recset, 'fields') and recset.fields:
        print("    Fields:")
        for field in recset.fields:
            field_type = getattr(field, 'data_type', None)
            print(f"      - Field @id: {field.id}   (Name: {field.name}, Data type: {field_type})")


## 3. Data Extraction

Load data from the record set(s) into Pandas DataFrames for exploration. Each record set is referenced by its `@id` field as found above.

We'll demonstrate with the principal record set for the protein analysis table. _If the dataset defines multiple record sets, you can loop across them as below._

In [ ]:
# List all record set IDs
record_set_ids = [r.id for r in metadata.record_sets]
print(f"Found the following record sets: {record_set_ids}\n")

# We will try loading the first record set for demonstration
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records from record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    print(f"  Loaded DataFrame with shape {df.shape}. Columns: {list(df.columns)}\n")
    dataframes[recset_id] = df

# Choose one for EDA below
main_record_set_id = record_set_ids[0]
print(f"We will use record set: {main_record_set_id} for further analysis.\n")
print(f"First five records:\n{dataframes[main_record_set_id].head()}")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate a typical workflow for a protein abundance analysis dataset: Filter by high-abundance proteins, normalize a measurement, and group by another field (if available).

All field names are referenced by their corresponding `@id` fields. Identify a numeric field to process (such as protein abundance, counts, or coverage percentage) and a group-by field.


In [ ]:
# Inspect available columns (these are usually mapped 1:1 from field @id)
df = dataframes[main_record_set_id]
print("Columns in current DataFrame:")
print(list(df.columns))

# Select a numeric field for analysis. For this dataset, likely candidates are 'Coverage(%)', 'MW', or 'Abundance' related columns
# Let's automatically attempt a match by searching for common abundance or quantitative column IDs or fallback to the first float column
import re
numeric_column = None
for col in df.columns:
    if re.search(r"abundance|Coverage|MW|count|peptide|intensity|amount|area|quantity|score", col, re.IGNORECASE):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_column = col
            break
# If not found by pattern, just select the first numeric column
if numeric_column is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_column = col
            break
if numeric_column is None:
    raise ValueError("No obvious numeric field for analysis found in the table.")
print(f"Using numeric field for EDA: {numeric_column}\n")

# Simple threshold filtering: proteins/rows with high numeric value
threshold = df[numeric_column].mean() + df[numeric_column].std()
filtered_df = df[df[numeric_column] > threshold]
print(f"Filtered records where {numeric_column} > {threshold:.2f} (mean + 1 std): {len(filtered_df)} records")
filtered_df = filtered_df.copy()

# Normalize the numeric field for filtered records
filtered_df[f"{numeric_column}_normalized"] = (
    filtered_df[numeric_column] - filtered_df[numeric_column].mean()
) / filtered_df[numeric_column].std()
print(f"\nFirst five normalized values:")
print(filtered_df[[numeric_column, f"{numeric_column}_normalized"]].head())

# Try to select a group-by field from the schema: common candidate would be `Description` or a sample/treatment field
group_field = None
for col in df.columns:
    if re.search(r"description|sample|treatment|group|class|type|category|condition", col, re.IGNORECASE):
        if pd.api.types.is_string_dtype(df[col]):
            group_field = col
            break
if group_field:
    print(f"\nGrouping by field: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_column].mean().reset_index()
    print("Mean value by group (first 5 rows):")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of our chosen numeric field and (if available) by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field (histogram)
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_column].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_column}")
plt.xlabel(numeric_column)
plt.ylabel('Count')
plt.show()

# If group_field is defined, plot groupwise mean
if group_field:
    plt.figure(figsize=(10,5))
    grouped = df.groupby(group_field)[numeric_column].mean().sort_values(ascending=False)
    sns.barplot(y=grouped.index, x=grouped.values, color='salmon')
    plt.title(f"Mean {numeric_column} by {group_field}")
    plt.xlabel(f"Mean {numeric_column}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to programmatically explore a tabular Croissant dataset using `mlcroissant`.
- Fields and columns are referenced by `@id`, making data access robust to schema evolution
- We loaded and inspected all record sets, extracted data to DataFrames, filtered and normalized protein attributes, and visualized their distributions

**Key observations:**
- _The full data structure can be browsed using the record set and field listing in Section 2_
- _Abundance metrics and protein characteristics can be analyzed after loading with only a few lines of Python using the schema's structure_

_For deeper domain analysis, consult the dataset's schema, documentation fields, and use additional metadata as needed._